In [3]:
import os
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
item_paths = os.path.join("../data/raw/ml-100k" , "u.item")

In [5]:
items = pd.read_csv(
    item_paths,
    sep = "|",
    header = None,
    encoding = "latin-1",
    usecols= [0, 1],
    names=["itemId", "title"]
)
items.head()

,itemId,title
0,1,Toy Story (1995)
1,2,GoldenEye (1995)
2,3,Four Rooms (1995)
3,4,Get Shorty (1995)
4,5,Copycat (1995)


In [6]:
# Drop rows where title is missing
items = items.dropna(subset=['title'])

In [7]:
# Create TF-IDF matrix
max_features = 5000
tfidf = TfidfVectorizer(stop_words='english', max_features=max_features)
tfidf_matrix = tfidf.fit_transform(items['title'])
sim_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [8]:
items.shape

(1682, 2)

In [ ]:
# Create a simple recommender class
class MovieRecommenderModel:
    def __init__(self, items, sim_matrix):
        self.item_lookup = items
        self.sim_matrix = sim_matrix
    
    def predict(self, model_input:pd.DataFrame):
        results = []
        for _, row in model_input.iterrows():
            query = str(row.get("title", ""))
            top_k = int(row.get("top_k", 5))

            matches = self.item_lookup[
                self.item_lookup['title'].str.contains(query, case=False, na=False)
            ]

            if len(matches) == 0:
                results.append([])
                continue

            idx = matches.index[0]
            sim_scores = list(enumerate(self.sim_matrix[idx]))
            sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
            top_indices = [i for i, s in sim_scores[1 : top_k + 1]]
            recommended = self.item_lookup.iloc[top_indices]['title'].tolist()
            results.append(recommended)
        return results 

In [10]:
input_df = pd.DataFrame([
    {"title": "Toy Story", "top_k": 5},
    {"title": "Jumanji", "top_k": 5},   
])
input_df.head()

,title,top_k
0,Toy Story,5
1,Jumanji,5


In [14]:
model = MovieRecommenderModel(items = items, sim_matrix = sim_matrix)
predictions = model.predict(input_df)
predictions

[["Pyromaniac's Love Story, A (1995)",
  'Story of Xinghua, The (1993)',
  'Philadelphia Story, The (1940)',
  'NeverEnding Story III, The (1994)',
  'FairyTale: A True Story (1997)'],
 ['Now and Then (1995)',
  'Show, The (1995)',
  'To Have, or Not (1995)',
  'Boys on the Side (1995)',
  'Murder in the First (1995)']]